# Task 1: CatBoost on MSRank LETOR Data

In [1]:
# Cell 1: Imports
from catboost import CatBoostRanker, Pool
import numpy as np
import pandas as pd
from copy import deepcopy
print('Imports done')

Imports done


In [2]:
# Cell 2: Load MSRank 10k dataset
from catboost.datasets import msrank_10k
train_df, test_df = msrank_10k()
print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Columns sample:', train_df.columns[:5].tolist())
print('Relevance label counts:\n', train_df[0].value_counts().sort_index())

Train shape: (10000, 138)
Test shape: (10000, 138)
Columns sample: [0, 1, 2, 3, 4]
Relevance label counts:
 0
0.0    5481
1.0    3000
2.0    1326
3.0     142
4.0      51
Name: count, dtype: int64


In [3]:
# Cell 3: Prepare features, labels, query IDs
X_train = train_df.drop([0, 1], axis=1).values
y_train = train_df[0].values / 4.0  # normalize to [0,1]
queries_train = train_df[1].values

X_test = test_df.drop([0, 1], axis=1).values
y_test = test_df[0].values / 4.0
queries_test = test_df[1].values

print(f'Features: {X_train.shape[1]}')
print(f'Train docs: {len(X_train)}, unique queries: {len(np.unique(queries_train))}')
print(f'Test docs:  {len(X_test)},  unique queries: {len(np.unique(queries_test))}')
print(f'Label range: [{y_train.min():.2f}, {y_train.max():.2f}]')

Features: 136
Train docs: 10000, unique queries: 87
Test docs:  10000,  unique queries: 88
Label range: [0.00, 1.00]


In [4]:
# Cell 4: Create CatBoost Pools
train_pool = Pool(data=X_train, label=y_train, group_id=queries_train)
test_pool  = Pool(data=X_test,  label=y_test,  group_id=queries_test)
print('Pools created')

Pools created


In [5]:
# Cell 5: Training helper and default parameters
default_parameters = {
    'iterations': 500,
    'custom_metric': ['NDCG', 'PFound', 'AverageGain:top=10'],
    'verbose': 100,
    'random_seed': 0,
}

models = {}

def fit_model(loss_function, extra_params=None):
    params = deepcopy(default_parameters)
    params['loss_function'] = loss_function
    if extra_params:
        params.update(extra_params)
    model = CatBoostRanker(**params)
    model.fit(train_pool, eval_set=test_pool)
    models[loss_function] = model
    print(f'[{loss_function}] done')
    return model

print('Helper defined')

Helper defined


In [6]:
# Cell 6: Train RMSE (regression baseline)
fit_model('RMSE')

Learning rate set to 0.111564
0:	learn: 0.1982058	test: 0.1965728	best: 0.1965728 (0)	total: 72.9ms	remaining: 36.4s


/Users/uni/LAlo/llama_env/lib/python3.12/site-packages/catboost/core.py:6705: RuntimeWarning: Regression loss ('RMSE') ignores an important ranking parameter 'group_id'
  warnings.warn("Regression loss ('{}') ignores an important ranking parameter 'group_id'".format(loss_function), RuntimeWarning)


100:	learn: 0.1662293	test: 0.1830727	best: 0.1829629 (85)	total: 501ms	remaining: 1.98s


200:	learn: 0.1529035	test: 0.1835774	best: 0.1829629 (85)	total: 894ms	remaining: 1.33s


300:	learn: 0.1429524	test: 0.1840282	best: 0.1829629 (85)	total: 1.29s	remaining: 851ms


400:	learn: 0.1345079	test: 0.1848505	best: 0.1829629 (85)	total: 1.68s	remaining: 416ms


499:	learn: 0.1267549	test: 0.1856600	best: 0.1829629 (85)	total: 2.07s	remaining: 0us

bestTest = 0.1829629431
bestIteration = 85

Shrink model to first 86 iterations.
[RMSE] done


CatBoostRanker(custom_metric=['NDCG', 'PFound', 'AverageGain:top=10'], iterations=500, loss_function='RMSE', random_seed=0, verbose=100)

In [7]:
# Cell 7: Train QueryRMSE
fit_model('QueryRMSE')

0:	learn: 0.1913124	test: 0.1842455	best: 0.1842455 (0)	total: 6.07ms	remaining: 3.03s


100:	learn: 0.1752932	test: 0.1769228	best: 0.1769228 (100)	total: 418ms	remaining: 1.65s


200:	learn: 0.1697477	test: 0.1757794	best: 0.1757794 (200)	total: 812ms	remaining: 1.21s


300:	learn: 0.1658221	test: 0.1753826	best: 0.1753826 (300)	total: 1.2s	remaining: 792ms


400:	learn: 0.1620817	test: 0.1750947	best: 0.1750947 (400)	total: 1.59s	remaining: 392ms


499:	learn: 0.1590064	test: 0.1750794	best: 0.1750781 (410)	total: 1.97s	remaining: 0us

bestTest = 0.1750781245
bestIteration = 410

Shrink model to first 411 iterations.
[QueryRMSE] done


CatBoostRanker(custom_metric=['NDCG', 'PFound', 'AverageGain:top=10'], iterations=500, loss_function='QueryRMSE', random_seed=0, verbose=100)

In [8]:
# Cell 8: Train PairLogit
fit_model('PairLogit')

0:	learn: 0.6913301	test: 0.6920109	best: 0.6920109 (0)	total: 64.8ms	remaining: 32.3s


100:	learn: 0.6119626	test: 0.6530342	best: 0.6530067 (99)	total: 6.72s	remaining: 26.5s


200:	learn: 0.5831140	test: 0.6488625	best: 0.6488086 (192)	total: 13.8s	remaining: 20.6s


300:	learn: 0.5610864	test: 0.6473297	best: 0.6472810 (299)	total: 21.2s	remaining: 14s


400:	learn: 0.5279680	test: 0.6395034	best: 0.6395034 (400)	total: 28.8s	remaining: 7.11s


499:	learn: 0.4732153	test: 0.6360252	best: 0.6352117 (468)	total: 36.7s	remaining: 0us

bestTest = 0.6352117251
bestIteration = 468

Shrink model to first 469 iterations.
[PairLogit] done


CatBoostRanker(custom_metric=['NDCG', 'PFound', 'AverageGain:top=10'], iterations=500, loss_function='PairLogit', random_seed=0, verbose=100)

In [9]:
# Cell 9: Train PairLogitPairwise
fit_model('PairLogitPairwise')

0:	learn: 0.6883382	test: 0.6896657	best: 0.6896657 (0)	total: 423ms	remaining: 3m 30s


100:	learn: 0.5479493	test: 0.6356205	best: 0.6356098 (97)	total: 46.4s	remaining: 3m 3s


200:	learn: 0.4950480	test: 0.6329879	best: 0.6329879 (200)	total: 1m 34s	remaining: 2m 20s


300:	learn: 0.4537769	test: 0.6343652	best: 0.6329087 (203)	total: 2m 23s	remaining: 1m 34s


400:	learn: 0.4186154	test: 0.6361870	best: 0.6329087 (203)	total: 3m 13s	remaining: 47.8s


499:	learn: 0.3884210	test: 0.6370577	best: 0.6329087 (203)	total: 4m 3s	remaining: 0us

bestTest = 0.632908695
bestIteration = 203

Shrink model to first 204 iterations.
[PairLogitPairwise] done


CatBoostRanker(custom_metric=['NDCG', 'PFound', 'AverageGain:top=10'], iterations=500, loss_function='PairLogitPairwise', random_seed=0, verbose=100)

In [10]:
# Cell 10: Train YetiRank (lr=0.3 to avoid underfitting)
fit_model('YetiRank', {'learning_rate': 0.3, 'train_dir': 'YetiRank_lr03'})

0:	test: 0.5488205	best: 0.5488205 (0)	total: 8.67ms	remaining: 4.33s


100:	test: 0.6606433	best: 0.6896435 (25)	total: 826ms	remaining: 3.26s


200:	test: 0.6745977	best: 0.6896435 (25)	total: 1.68s	remaining: 2.5s


300:	test: 0.6626918	best: 0.6896435 (25)	total: 2.5s	remaining: 1.66s


400:	test: 0.6675856	best: 0.6896435 (25)	total: 3.38s	remaining: 836ms


499:	test: 0.6643642	best: 0.6896435 (25)	total: 4.19s	remaining: 0us

bestTest = 0.6896435471
bestIteration = 25

Shrink model to first 26 iterations.
[YetiRank] done


CatBoostRanker(custom_metric=['NDCG', 'PFound', 'AverageGain:top=10'], iterations=500, learning_rate=0.3, loss_function='YetiRank', random_seed=0, train_dir='YetiRank_lr03', verbose=100)

In [11]:
# Cell 11: Train YetiRankPairwise
fit_model('YetiRankPairwise', {'learning_rate': 0.3, 'train_dir': 'YetiRankPairwise_lr03'})

0:	test: 0.5219547	best: 0.5219547 (0)	total: 40.1ms	remaining: 20s


100:	test: 0.6896125	best: 0.6932658 (93)	total: 4.13s	remaining: 16.3s


200:	test: 0.6936051	best: 0.6973380 (152)	total: 8.24s	remaining: 12.3s


300:	test: 0.6932663	best: 0.6973380 (152)	total: 12.3s	remaining: 8.14s


400:	test: 0.6934867	best: 0.6973380 (152)	total: 16.4s	remaining: 4.04s


499:	test: 0.6922567	best: 0.6973380 (152)	total: 20.4s	remaining: 0us

bestTest = 0.6973380275
bestIteration = 152

Shrink model to first 153 iterations.
[YetiRankPairwise] done


CatBoostRanker(custom_metric=['NDCG', 'PFound', 'AverageGain:top=10'], iterations=500, learning_rate=0.3, loss_function='YetiRankPairwise', random_seed=0, train_dir='YetiRankPairwise_lr03', verbose=100)

In [12]:
# Cell 12: Collect and report results
results = []
for loss_fn, model in models.items():
    metrics = model.eval_metrics(test_pool, ['NDCG', 'PFound', 'AverageGain:top=10'])
    # inspect actual keys
    keys = list(metrics.keys())
    ndcg_key     = [k for k in keys if 'NDCG' in k][0]
    pfound_key   = [k for k in keys if 'PFound' in k][0]
    avg_gain_key = [k for k in keys if 'AverageGain' in k][0]
    results.append({
        'Loss Function':  loss_fn,
        'NDCG':           metrics[ndcg_key][-1],
        'PFound':         metrics[pfound_key][-1],
        'AverageGain@10': metrics[avg_gain_key][-1],
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('NDCG', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

    Loss Function     NDCG   PFound  AverageGain@10
        QueryRMSE 0.752336 0.701750        0.259659
        PairLogit 0.748523 0.699989        0.259375
             RMSE 0.746492 0.696391        0.256534
 YetiRankPairwise 0.746430 0.697338        0.260227
PairLogitPairwise 0.745145 0.687449        0.258807
         YetiRank 0.740875 0.689644        0.250568
